In [ ]:
#copied preprocessing method form 02
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from PIL import Image

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

BASE_DIR = "../data"
#pls keep the data file at where yall put like in the 01 note, ty!
TRAIN_DIR = os.path.join(BASE_DIR, "Train")
VAL_DIR = os.path.join(BASE_DIR, "Validation")
TEST_DIR = os.path.join(BASE_DIR, "Test")

#i recommend going with image size 128 square and batch size 32 like many examples in class
IMG_SIZE = (128, 128)
BATCH_SIZE = 32

#load dataset (and under the form of RGB right away too)
#we also will resize the img into IMG-_SIZE of 128 square too
train_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary",
    color_mode="rgb",
    shuffle=True,
    seed=SEED
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    VAL_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary",
    color_mode="rgb",
    shuffle=False
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary",
    color_mode="rgb",
    shuffle=False
)

#check number of img and class name
print(train_ds.class_names)
print('-'*10)

#normalize (make the RGB valuables from 0-255 range to 0-1 range)
#it's basically just a devide math, new pixel num = old num : 255
normalization_layer = tf.keras.layers.Rescaling(1./255)

train_ds_nor = train_ds.map(lambda images, labels: (normalization_layer(images), labels))
val_ds_nor = val_ds.map(lambda images, labels: (normalization_layer(images), labels))
test_ds_nor = test_ds.map(lambda images, labels: (normalization_layer(images), labels))
#get a batch from pre-normalized set
raw_images, raw_labels = next(iter(train_ds))
#and one from normalized set
normalized_images, normalized_labels = next(iter(train_ds_nor))

print("Before normalization:")
print("Shape:", raw_images.shape)
print("Min val:", tf.reduce_min(raw_images).numpy())
print("Max val:", tf.reduce_max(raw_images).numpy())

print("\nAfter normalization:")
print("Shape:", normalized_images.shape)
print("Min val:", tf.reduce_min(normalized_images).numpy())
print("Max val:", tf.reduce_max(normalized_images).numpy())
print('-'*10)

#augmentation (but on normalized sets)
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.1)
], name="data_augmentation")

train_ds_augmented = train_ds_nor.map(lambda x, y: (data_augmentation(x, training=True), y))
print('data has been augmented!')

#check aug
images, labels = next(iter(train_ds_nor))
augmented_images = data_augmentation(
    images,
    training=True
)

plt.figure(figsize=(10, 5))

for i in range(5):
    plt.subplot(2, 5, i + 1)
    plt.imshow(images[i])
    plt.title("Original")
    plt.axis("off")

    plt.subplot(2, 5, i + 6)
    plt.imshow(augmented_images[i])
    plt.title("Augmented")
    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

#load img to ram right away once, save time and cpu power. 
#cuz cpu will load img in EVERY epochs without this. really sad if it really have to do so.

train_ds_final = train_ds_augmented.cache().prefetch(buffer_size=AUTOTUNE)

# val and test no need augmentation
val_ds_final = val_ds_nor.cache().prefetch(buffer_size=AUTOTUNE)
test_ds_final = test_ds_nor.cache().prefetch(buffer_size=AUTOTUNE)

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Conv2D, MaxPooling2D, BatchNormalization, 
    Dropout, Flatten, Dense
)

# Initialize a heavy-duty Sequential model
model = Sequential([
    # --- Block 1: Low-level features (Input: 128x128x3) ---
    Conv2D(64, (3, 3), activation='relu', padding='same', input_shape=(128, 128, 3)),
    BatchNormalization(),
    Conv2D(64, (3, 3), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling2D(pool_size=(2, 2)), # Down to 64x64
    Dropout(0.25),

    # --- Block 2: Mid-level features ---
    Conv2D(128, (3, 3), activation='relu', padding='same'),
    BatchNormalization(),
    Conv2D(128, (3, 3), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling2D(pool_size=(2, 2)), # Down to 32x32
    Dropout(0.25),

    # --- Block 3: Complex Pattern Recognition ---
    Conv2D(256, (3, 3), activation='relu', padding='same'),
    BatchNormalization(),
    Conv2D(256, (3, 3), activation='relu', padding='same'),
    BatchNormalization(),
    Conv2D(256, (3, 3), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling2D(pool_size=(2, 2)), # Down to 16x16
    Dropout(0.3),

    # --- Block 4: Deep Semantic Features ---
    Conv2D(512, (3, 3), activation='relu', padding='same'),
    BatchNormalization(),
    Conv2D(512, (3, 3), activation='relu', padding='same'),
    BatchNormalization(),
    Conv2D(512, (3, 3), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling2D(pool_size=(2, 2)), # Down to 8x8
    Dropout(0.3),

    # --- Block 5: High-Level Abstraction ---
    Conv2D(512, (3, 3), activation='relu', padding='same'),
    BatchNormalization(),
    Conv2D(512, (3, 3), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling2D(pool_size=(2, 2)), # Down to 4x4
    Dropout(0.4),

    # --- Massive Classification Head ---
    Flatten(),
    Dense(512, activation='relu'),
    BatchNormalization(),
    Dropout(0.5),
    
    Dense(256, activation='relu'),
    BatchNormalization(),
    Dropout(0.5),

    # Output Layer for Binary Classification (Mask vs No Mask)
    Dense(1, activation='sigmoid')
])

# Compile the model with all your metrics
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=[
        'accuracy', 
        tf.keras.metrics.Precision(name='precision'), 
        tf.keras.metrics.Recall(name='recall'), 
    ]
)

# Display model summary
model.summary()

In [ ]:
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping

# 1. Define Callbacks to optimize training
reduce_lr = ReduceLROnPlateau(
    monitor='val_loss', 
    factor=0.5,           # Cut learning rate in half if validation loss stops improving
    patience=3,           # Wait 3 epochs before reducing
    min_lr=1e-6, 
    verbose=1
)

early_stop = EarlyStopping(
    monitor='val_loss', 
    patience=10,          # Stop training if validation loss doesn't improve for 10 epochs
    restore_best_weights=True, # Keep the weights from the absolute best epoch
    verbose=1
)

# 2. Set number of epochs
EPOCHS = 50  # 20-30 epochs is usually plenty for a mask dataset with data augmentation

# 3. Train the Model
history = model.fit(
    train_ds_final,          # Your cached, prefetched, and augmented training data
    validation_data=val_ds_final, # Your validation data
    epochs=EPOCHS,
    callbacks=[reduce_lr, early_stop],
    verbose=1
)

In [ ]:
# Save the trained model to a file
model.save("../model/mask_complex_CNN_01.keras")
print("Model successfully saved to '../model/mask_complex_CNN_01.keras'!")